## Retrieval-Augmented Generation (RAG)

**Retrieval-Augmented Generation (RAG)** lets Claude answer questions using a large document it never saw at training time and can't fit into a single prompt. Instead of pasting the whole document into every request, we search it first and hand Claude only the relevant pieces.

The pipeline has four stages, and this notebook covers each one as its own section:

1. **Chunking** — split a large document into smaller, topically-focused pieces.
2. **Embeddings** — turn each chunk into a vector (a list of numbers) that represents its meaning.
3. **Vector DB** — store every chunk's vector so it can be searched later.
4. **Semantic Search** — embed a question and find the chunks whose meaning is closest to it, then hand those to Claude.

We use Voyage AI for embeddings (Claude itself doesn't generate embeddings) and Anthropic's Claude for the final answer. `report.md` — a 15-section fictional annual report — is the running example document.

In [1]:
%pip install -q anthropic voyageai python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Setup: load environment variables and create both clients
from dotenv import load_dotenv
from anthropic import Anthropic
import voyageai

load_dotenv()

anthropic_client = Anthropic()
model = "claude-haiku-5-5"

# DB C [client module] /S [remote]

voyage_client = voyageai.Client()

## 1. Chunking

**Definition:** Chunking is splitting a large document into smaller pieces ("chunks") before storing it for retrieval. RAG works by finding the *most relevant* pieces of a document and handing only those to Claude — but that only helps if a chunk is small enough to be about one topic. A chunk that's too big drags in irrelevant context; a chunk that's too small loses the surrounding meaning.

There are a few ways to decide where the splits go:

- **By character count** — cut every N characters, with a little overlap so a split doesn't destroy a sentence at the boundary. Simple, but can still cut mid-sentence.
- **By sentence** — group a fixed number of sentences per chunk. Keeps prose intact, but chunk sizes vary.
- **By document structure** — split on the document's own headings (e.g. every `## ` in Markdown). If the author already organized the document into topics, this gives you one chunk per topic for free — the cleanest split when the structure is available.

We'll use structure-based chunking as the running example for the rest of this notebook, since `report.md` is already organized into headed sections.

In [3]:
def chunk_by_section(document_text):
    import re

    pattern = r"\n## "
    return re.split(pattern, document_text)

In [4]:
with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)

print(f"{len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    preview = chunk[:80].replace("\n", " ")
    print(f"[{i}] {len(chunk):>5} chars | {preview}...")

15 chunks

[0]    70 chars | # **Annual Interdisciplinary Research Review: Cross-Domain Insights** ...
[1]  1960 chars | Executive Summary  This report synthesizes the key findings and ongoing research...
[2]   854 chars | Table of Contents  1.  Executive Summary 2.  Table of Contents 3.  Methodology 4...
[3]  1135 chars | Methodology  The insights compiled within this Annual Interdisciplinary Research...
[4]  1178 chars | Section 1: Medical Research - Understanding XDR-471 Syndrome  This year saw sign...
[5]  1243 chars | Section 2: Software Engineering - Project Phoenix Stability Enhancements  The So...
[6]  1170 chars | Section 3: Financial Analysis - Q3 Performance and Outlook  Quarterly financial ...
[7]  1181 chars | Section 4: Scientific Experimentation - Characterization of Material Composite X...
[8]  1223 chars | Section 5: Legal Developments - Navigating IP Precedents and Regulatory Shifts  ...
[9]  1142 chars | Section 6: Product Engineering - Finalizing Model Zircon-5 Spec

### Other chunking strategies

For reference, here are the character-based and sentence-based strategies mentioned above. Both take a block of text and return a list of chunk strings, the same shape as `chunk_by_section` — swap them in when a document has no clear heading structure to split on.

In [ ]:
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    chunks = []
    start_idx = 0

    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))
        chunks.append(text[start_idx:end_idx])
        start_idx = end_idx - chunk_overlap if end_idx < len(text) else len(text)

    return chunks


def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    import re

    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    start_idx = 0

    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))
        chunks.append(" ".join(sentences[start_idx:end_idx]))
        start_idx = max(0, start_idx + max_sentences_per_chunk - overlap_sentences)

    return chunks

In [ ]:
char_chunks = chunk_by_char(text)
sentence_chunks = chunk_by_sentence(text)

print(f"chunk_by_char:     {len(char_chunks)} chunks")
print(f"chunk_by_sentence: {len(sentence_chunks)} chunks\n")

print("chunk_by_char[0]:")
print(repr(char_chunks[0]))

print("\nchunk_by_sentence[0]:")
print(repr(sentence_chunks[0]))

## 2. Embeddings

**Definition:** An embedding is a list of numbers (a vector) that represents the *meaning* of a piece of text. Texts with similar meaning get vectors that point in similar directions — this is what lets us search by meaning instead of by matching exact words.

We use Voyage AI's embedding model here via the `voyageai` client. `input_type` tells Voyage which side of a future search this text is on:

- `"document"` — for text we're storing (our chunks).
- `"query"` — for a question we'll search with.

The model encodes them slightly differently, and using the right one improves retrieval — we'll use both in sections 3 and 4.

In [5]:
def generate_embedding(chunks, model="voyage-4-lite", input_type="document"):
    is_list = isinstance(chunks, list)
    input = chunks if is_list else [chunks]
    result = voyage_client.embed(input, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

In [13]:
sample_text = "Compound CTX-204b showed promising results in its Phase 2a clinical trial."

embedding = generate_embedding(sample_text)

print(f"{len(embedding)} dimensions")
print("First 5 values:", embedding)

1024 dimensions
First 5 values: [-0.027578001841902733, -0.025960665196180344, -0.004285987000912428, 0.04466550424695015, 0.013972373679280281, -0.015627173706889153, -0.04347185045480728, -0.0004804774944204837, 0.05327099934220314, -0.03698821738362312, -0.003282854799181223, 0.028609860688447952, -0.018980147317051888, -0.000923922925721854, -0.003589956322684884, -0.04059702903032303, -0.038484785705804825, 0.029945528134703636, 0.01791391894221306, -0.02516653947532177, 0.0016400101594626904, 0.028557471930980682, -0.04269300773739815, -0.004324559587985277, 0.045667652040719986, -0.009631547145545483, -0.03523705154657364, 0.04129515588283539, 0.03274696692824364, -0.0294225811958313, 0.013107141479849815, 0.03426078334450722, 0.01351997908204794, 0.0037347569596022367, 0.06560459733009338, -0.00972569826990366, -0.002753345062956214, -0.06996908783912659, -0.0542415976524353, -0.029398487880825996, -0.013401550240814686, -0.004960944876074791, 0.03306096792221069, -0.0105858789

## 3. Vector DB

**Definition:** A vector database (or vector *index*, for our small-scale purposes) stores each chunk's embedding alongside the chunk itself, and answers a search by comparing a query's embedding against every stored one — returning the chunks whose meaning is closest to the query's meaning.

Here we build a minimal, dependency-free `VectorIndex`: two parallel lists (`vectors[i]` is the embedding of `documents[i]`), and a brute-force scan that scores the query against every stored vector. That's `O(n)` per search — fine for a handful of chunks. A production vector database (Pinecone, pgvector, Chroma, ...) swaps the brute-force scan for an approximate-nearest-neighbour index so it stays fast at millions of vectors, but the interface — add documents, search, get the closest matches back — is the same.

![RAG_Image](./images/rag.png)

In [7]:
# VectorIndex implementation
#
# A minimal, dependency-free stand-in for a real vector database. It keeps two
# parallel lists - self.vectors[i] is the embedding of self.documents[i] - and
# answers a search by scoring the query against every stored vector.
import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(self, distance_metric: str = "cosine", embedding_fn=None):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError("Embedding function not provided during initialization.")
        vector = self._embedding_fn(document["content"])
        self.add_vector(vector=vector, document=document)

    def add_documents(self, documents: List[Dict[str, Any]], vectors: Optional[List[List[float]]] = None):
        # Bulk version of add_document, to avoid one API call per chunk.
        if vectors is None:
            if not self._embedding_fn:
                raise ValueError("Provide either precomputed vectors or an embedding function.")
            vectors = self._embedding_fn([doc["content"] for doc in documents])

        for vector, document in zip(vectors, documents):
            self.add_vector(vector=vector, document=document)

    def add_vector(self, vector, document: Dict[str, Any]):
        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}")

        self.vectors.append(list(vector))
        self.documents.append(document)

    def search(self, query: Any, k: int = 1) -> List[Tuple[Dict[str, Any], float]]:
        # Accepts raw text (embedded on the fly) or a ready-made vector.
        # Returns (document, distance) pairs — lower distance is a better match.
        if not self.vectors:
            return []

        if isinstance(query, str):
            query_vector = self._embedding_fn(query)
        else:
            query_vector = query

        dist_func = self._cosine_distance if self._distance_metric == "cosine" else self._euclidean_distance

        distances = [(dist_func(query_vector, v), doc) for v, doc in zip(self.vectors, self.documents)]
        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def _euclidean_distance(self, vec1, vec2) -> float:
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _cosine_distance(self, vec1, vec2) -> float:
        # 1 - cosine similarity: 0 (identical direction) to 2 (opposite).
        # Compares direction only, so a long chunk and a short chunk about the
        # same topic still score close.
        mag1 = math.sqrt(sum(x * x for x in vec1))
        mag2 = math.sqrt(sum(x * x for x in vec2))
        if mag1 == 0 or mag2 == 0:
            return 1.0
        dot = sum(p * q for p, q in zip(vec1, vec2))
        similarity = max(-1.0, min(1.0, dot / (mag1 * mag2)))
        return 1.0 - similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}')"

### Building the index

Reuse the chunks from section 1, embed every chunk with `input_type="document"`, and store the vectors alongside the chunk text.

In [8]:
embeddings = generate_embedding(chunks, input_type="document")

store = VectorIndex()
store.add_documents([{"content": chunk} for chunk in chunks], vectors=embeddings)

store

VectorIndex(count=15, dim=1024, metric='cosine')

## 4. Semantic Search

**Definition:** Semantic search is embedding a question (`input_type="query"`) and asking the `VectorIndex` for the chunks whose vectors are closest to it. Because matching happens on meaning rather than keywords, a question that shares no words with a chunk can still be its nearest vector — that's the whole point of embedding-based retrieval over plain keyword search.

We search first, then hand the retrieved chunks to Claude as context so it can answer using only the relevant part of the document — this is the full RAG loop end to end.

In [9]:
query = "What did the Phase IIa trial of the new compound show?"
query_embedding = generate_embedding(query, input_type="query")

results = store.search(query_embedding, k=2)

for rank, (doc, distance) in enumerate(results, start=1):
    preview = doc["content"][:300].replace("\n", " ")
    print(f"--- Rank {rank} | distance {distance:.4f} ---")
    print(preview + "...\n")

--- Rank 1 | distance 0.4922 ---
Section 9: Pharmaceutical Development - Compound CTX-204b Phase IIa Update  Promising results emerged from the Phase IIa clinical trial (`Trial ID: CTX204b-P2A-001`) for Compound CTX-204b, our lead candidate targeting Receptor Pathway Gamma-7. Interim analysis of data from the initial patient cohort...

--- Rank 2 | distance 0.6809 ---
Section 1: Medical Research - Understanding XDR-471 Syndrome  This year saw significant strides in our understanding of XDR-471 syndrome, a rare neurodegenerative condition previously hampered by diagnostic ambiguity. The team focused on correlating clinical presentations with specific genetic marke...



### Answering with Claude

Finally, close the loop: pass the top retrieved chunk(s) to Claude as context in the prompt, and let Claude write the answer using only that retrieved text — not the whole document.

In [10]:
context = "\n\n---\n\n".join(doc["content"] for doc, _ in results)

prompt = f"""Use the following context to answer the question. If the context doesn't contain the answer, say so.

<context>
{context}
</context>

Question: {query}"""

response = anthropic_client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[{"role": "user", "content": prompt}],
)

next(block.text for block in response.content if block.type == "text")

'The Phase IIa trial of Compound CTX-204b (Trial ID CTX204b-P2A-001) showed:\n\n- **Efficacy:** In an interim analysis of the initial cohort (n=30), the compound produced a statistically significant change in the primary efficacy biomarker, BioMkr-HGF, compared to placebo (p=0.015).\n- **Safety:** The safety profile was generally consistent with Phase I. There was a slightly higher incidence of mild gastrointestinal adverse events (AE Code CTX-GI-002), which requires ongoing monitoring.\n\nThese interim results support moving forward to dose-range finding studies. The context does not give the full trial results, since this is an interim analysis of an initial cohort.'

In [ ]:
# Sample Prompts:
#
# Each one is picked to exercise a different RAG behavior against report.md:
#
# - "paraphrase"   : shares almost no words with the source chunk, so only
#                    semantic (not keyword) matching can find it.
# - "id_lookup"    : asks for a specific figure/ID that lives in exactly one
#                    chunk — tests precision, not just topical relevance.
# - "cross_section": the answer is split across two+ sections, so it only
#                    works if k >= 2 and both relevant chunks get retrieved.
# - "ambiguous_id" : `INC-2023-Q4-011` is reused in both Section 2 (software)
#                    and Section 10 (cybersecurity) for unrelated events -
#                    a good stress test for which chunk(s) get pulled back.
# - "out_of_scope" : nothing in report.md answers this - demonstrates that
#                    the prompt's "say so if the context doesn't contain the
#                    answer" instruction actually holds Claude back from
#                    guessing.

sample_prompts = [
    {
        "label": "paraphrase",
        "query": "How is the new material holding up under repeated stress cycles?",
        "k": 2,
    },
    {
        "label": "id_lookup",
        "query": "What CPU clock speed and memory were confirmed for the Zircon-5 prototype?",
        "k": 2,
    },
    {
        "label": "cross_section",
        "query": "Which other teams' work depends on the Software Engineering stability fixes?",
        "k": 3,
    },
    {
        "label": "ambiguous_id",
        "query": "What happened with INC-2023-Q4-011?",
        "k": 3,
    },
    {
        "label": "out_of_scope",
        "query": "How many total employees does the company have?",
        "k": 2,
    },
]

for item in sample_prompts:
    query_embedding = generate_embedding(item["query"], input_type="query")
    results = store.search(query_embedding, k=item["k"])

    context = "\n\n---\n\n".join(doc["content"] for doc, _ in results)
    rag_prompt = f"""Use the following context to answer the question. If the context doesn't contain the answer, say so.

<context>
{context}
</context>

Question: {item["query"]}"""

    response = anthropic_client.messages.create(
        model=model,
        max_tokens=500,
        messages=[{"role": "user", "content": rag_prompt}],
    )

    retrieved_idx = [chunks.index(doc["content"]) for doc, _ in results]
    print(f"=== [{item['label']}] {item['query']} ===")
    print(f"retrieved chunks: {retrieved_idx}")
    print(next(block.text for block in response.content if block.type == "text"))
    print()
